In [ ]:
import os
import sys
import argparse
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import plotly.io as pio
import plotly.offline as pyo

pio.renderers.default = 'iframe'
pyo.init_notebook_mode(connected=True)

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
mpl.rcParams['figure.dpi'] = 150
sns.set_palette("husl")

In [ ]:
# Read the file for urban factors, SWORD reaches, usgs gage heights
sword_urban = gpd.read_parquet(
    "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/usgs_sword_matchups_merritt_v2.parquet"
)

sword_urban = sword_urban.dropna(axis=1, how='all')

In [ ]:
sword_urban

In [ ]:
# Read the file for urban factors, SWORD reaches, usgs gage heights
sword_urban2 = gpd.read_parquet(
    "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/usgs_sword_matchups_merritt.parquet"
)

sword_urban2 = sword_urban2.dropna(axis=1, how='all')

In [ ]:
sword_urban2

In [ ]:
reaches_nlcd = sword_urban.copy()

In [ ]:
reaches_nlcd = reaches_nlcd.drop_duplicates(['reach_id','site_no','nlcd_21_mean', 'nlcd_22_mean','nlcd_23_mean', 'nlcd_24_mean','fctimp_mean'])
reaches_nlcd = reaches_nlcd.to_crs('EPSG:4326')
reaches_nlcd['fctimp_mean'] = reaches_nlcd['fctimp_mean'] / 100

print(f"CRS: {reaches_nlcd.crs}")
print(f"Bounds: {reaches_nlcd.total_bounds}")

In [ ]:
reaches_nlcd.to_parquet('/work/pi_kandread_umass_edu/swot-urban/1_data_processing/reaches_nlcd_merritt_v2.parquet')

In [ ]:
reaches_nlcd = gpd.read_parquet('/work/pi_kandread_umass_edu/swot-urban/1_data_processing/reaches_nlcd_merritt_v2.parquet')

In [ ]:
filtered_swot = pd.read_parquet('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/swotall_qc_filtered_vD.parquet')
filtered_swot_id = filtered_swot.reach_id.drop_duplicates().to_list()

In [ ]:
reaches_nlcd = reaches_nlcd[reaches_nlcd.reach_id.isin(filtered_swot_id)]
reaches_nlcd.reach_id.drop_duplicates().shape

In [ ]:
# fiona_sites = ['02233500', '02234990', '03122500', '03124000',
#                 '03130000', '03133500', '03135000', '07359002',
#                 '08016800', '08167000', '12113350']
# reaches_nlcd[reaches_nlcd.site_no.isin(fiona_sites)].to_file('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/fiona_sites.shp')

In [ ]:
"""
Urban Area Histograms with CDF
"""

def plot_hist_cdf(ax, data, col, title, color='steelblue'):
    """Plot histogram with CDF on twin axis"""
    vals = data[col].dropna()
    if len(vals) == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center')
        return
    
    # Histogram
    ax.hist(vals, bins=100, alpha=0.7, color=color, edgecolor='black', linewidth=0.5)
    ax.axvline(vals.median(), color='red', linestyle='--', linewidth=2, 
               label=f'Median: {vals.median():.3f}')
    ax.axvline(vals.mean(), color='orange', linestyle=':', linewidth=2, 
               label=f'Mean: {vals.mean():.3f}')
    ax.set_xlabel('Fraction', fontweight='bold')
    ax.set_ylabel('Number of Reaches', fontweight='bold')
    ax.set_title(title, fontweight='bold', fontsize=13)
    ax.grid(alpha=0.3)
    
    # CDF
    ax2 = ax.twinx()
    sorted_vals = np.sort(vals)
    cdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
    ax2.plot(sorted_vals, cdf, 'darkgreen', linewidth=2.5, alpha=0.8, label='CDF')
    ax2.set_ylabel('CDF', fontweight='bold', color='darkgreen')
    ax2.tick_params(axis='y', labelcolor='darkgreen')
    ax2.set_ylim(-0.05, 1.05)
    
    # # Stats box
    # stats = f'n={len(vals):,}\nμ={vals.mean():.3f}\nσ={vals.std():.3f}\nMed={vals.median():.3f}'
    # ax.text(0.98, 0.97, stats, transform=ax.transAxes, fontsize=9, 
    #         va='top', ha='right', family='monospace',
    #         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    # Legend
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='center right', framealpha=0.9)


def create_urban_histograms(data, save_path=None):
    """    
    Parameters:
    -----------
    data : DataFrame
        Data with urban area columns
    save_path : str, optional
        Path to save figure
    
    Returns:
    --------
    fig : matplotlib figure
    stats_df : DataFrame with statistics
    """
    
    # Plot configuration
    plots = [
        ('fctimp_mean', 'Total Impervious Surface', 'steelblue', (0, slice(0, 2))),
        ('nlcd_21_mean', 'Open Space', '#7fc97f', (1, 0)),
        ('nlcd_22_mean', 'Low Intensity', '#beaed4', (1, 1)),
        ('nlcd_23_mean', 'Medium Intensity', '#fdc086', (2, 0)),
        ('nlcd_24_mean', 'High Intensity', '#e41a1c', (2, 1))
    ]
    
    # Create figure
    fig = plt.figure(figsize=(16, 14))
    gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.35, wspace=0.3)
    
    # Generate plots
    stats_list = []
    for col, title, color, pos in plots:
        if col not in data.columns:
            print(f"Warning: {col} not found, skipping")
            continue
        
        # Create subplot
        if isinstance(pos[1], slice):
            ax = fig.add_subplot(gs[pos[0], pos[1]])
        else:
            ax = fig.add_subplot(gs[pos[0], pos[1]])
        
        # Plot
        plot_hist_cdf(ax, data, col, title, color)
        
        # Collect statistics
        vals = data[col].dropna()
        if len(vals) > 0:
            stats_list.append({
                'class': title,
                'column': col,
                'n': len(vals),
                'mean': vals.mean(),
                'median': vals.median(),
                'std': vals.std(),
                'q25': vals.quantile(0.25),
                'q75': vals.quantile(0.75)
            })
    
    # Save if path provided
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"Saved figure: {save_path}")
        
        # Save statistics
        stats_file = save_path.replace('.png', '_stats.csv')
        stats_df = pd.DataFrame(stats_list)
        stats_df.to_csv(stats_file, index=False, float_format='%.4f')
        print(f"Saved stats: {stats_file}")
    
    # Create stats DataFrame
    stats_df = pd.DataFrame(stats_list)
    
    # Display statistics
    print("\nStatistics Summary:")
    print(stats_df.to_string(index=False))
    
    return fig, stats_df

fig, stats = create_urban_histograms(reaches_nlcd, save_path='/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/histograms_reaches_urban.png')
plt.show()

In [ ]:
def plot_urban_grid(gdf, save_path=None):
    """Create 3x2 grid of urban maps with full features"""
    
    gdf = gdf.to_crs(epsg=4326).copy()
    gdf['centroid'] = gdf.geometry.centroid
    
    # Load states once
    try:
        states = gpd.read_file('/home/kwanhyuckkim_umass_edu/back_folder/tl_2012_us_state.shp')
        states = states.to_crs(epsg=4326).cx[-170:-50, 12:85]
        has_states = True
    except:
        has_states = False
    
    # Configuration
    maps = [
        ('fctimp_mean', 'Total Impervious Surface', 'RdPu'),
        ('nlcd_21_mean', 'Open Space', 'Greens'),
        ('nlcd_22_mean', 'Low Intensity Development', 'Blues'),
        ('nlcd_23_mean', 'Medium Intensity Development', 'Oranges'),
        ('nlcd_24_mean', 'High Intensity Development', 'Reds'),
    ]
    
    # Create 3x2 figure with black background
    fig = plt.figure(figsize=(14, 18), facecolor='black')
    
    for idx, (col, title, cmap_name) in enumerate(maps, 1):
        ax = plt.subplot(3, 2, idx, projection=ccrs.PlateCarree())
        ax.set_facecolor('#1a1a1a')
        
        # Add cartopy features
        ax.add_feature(cfeature.LAND.with_scale('50m'), color='#2a2a2a')
        ax.add_feature(cfeature.COASTLINE.with_scale('50m'), linewidth=0.5, edgecolor='#666666')
        ax.add_feature(cfeature.BORDERS.with_scale('50m'), linewidth=0.5, edgecolor='#666666')
        ax.add_feature(cfeature.LAKES.with_scale('50m'), color='#1a3a52')
        
        # Add state lines
        if has_states:
            ax.add_geometries(states.geometry, ccrs.PlateCarree(), 
                             edgecolor='#555555', facecolor='none', linewidth=0.5)
        
        # Get actual data range
        vals = gdf[col].dropna().values
        vmin = 0
        vmax = vals.max()  # Use actual maximum!
        
        # Plot data
        sc = ax.scatter(gdf.centroid.x, gdf.centroid.y, 
                       c=gdf[col].values, cmap=cmap_name, s=10, alpha=0.8,
                       vmin=vmin, vmax=vmax, edgecolor='none',
                       transform=ccrs.PlateCarree())
        
        # Set extent
        ax.set_extent([-130, -65, 24, 50], crs=ccrs.PlateCarree())
        
        # Gridlines
        gl = ax.gridlines(draw_labels=True, linewidth=0.5, 
                         color='#444444', alpha=0.5, linestyle='--')
        gl.top_labels = gl.right_labels = False
        gl.xlabel_style = {'size': 9, 'color': 'white'}
        gl.ylabel_style = {'size': 9, 'color': 'white'}
        
        # Title
        ax.set_title(title, fontsize=12, fontweight='bold', pad=8, color='white')
        
        # Colorbar with actual range
        cbar = plt.colorbar(sc, ax=ax, orientation='horizontal', 
                           pad=0.05, shrink=0.9, aspect=25)
        
        # Create ticks based on actual max
        ticks = [0, vmax*0.2, vmax*0.4, vmax*0.6, vmax*0.8, vmax]
        tick_labels = ['0%', f'{vmax*20:.0f}%', f'{vmax*40:.0f}%', 
                      f'{vmax*60:.0f}%', f'{vmax*80:.0f}%', f'{vmax*100:.0f}%']
        
        cbar.set_ticks(ticks)
        cbar.set_ticklabels(tick_labels)
        cbar.ax.tick_params(labelsize=9, colors='white')
        cbar.outline.set_edgecolor('white')
    
    # Add 6th panel: Dominant type
    ax = plt.subplot(3, 2, 6, projection=ccrs.PlateCarree())
    ax.set_facecolor('#1a1a1a')
    
    # Add cartopy features
    ax.add_feature(cfeature.LAND.with_scale('50m'), color='#2a2a2a')
    ax.add_feature(cfeature.COASTLINE.with_scale('50m'), linewidth=0.5, edgecolor='#666666')
    ax.add_feature(cfeature.BORDERS.with_scale('50m'), linewidth=0.5, edgecolor='#666666')
    ax.add_feature(cfeature.LAKES.with_scale('50m'), color='#1a3a52')
    
    # Add state lines
    if has_states:
        ax.add_geometries(states.geometry, ccrs.PlateCarree(), 
                         edgecolor='#555555', facecolor='none', linewidth=0.5)
    
    # Dominant type
    urban_cols = ['nlcd_21_mean', 'nlcd_22_mean', 'nlcd_23_mean', 'nlcd_24_mean']
    gdf['dominant'] = gdf[urban_cols].idxmax(axis=1).map({
        'nlcd_21_mean': 0, 'nlcd_22_mean': 1, 
        'nlcd_23_mean': 2, 'nlcd_24_mean': 3
    })
    
    from matplotlib.colors import ListedColormap
    colors_dominant = ['#7fc97f', '#beaed4', '#fdc086', '#e41a1c']
    cmap_dominant = ListedColormap(colors_dominant)
    
    sc = ax.scatter(gdf.centroid.x, gdf.centroid.y, 
                   c=gdf['dominant'], cmap=cmap_dominant, s=10, alpha=0.8,
                   vmin=0, vmax=3, edgecolor='none',
                   transform=ccrs.PlateCarree())
    
    ax.set_extent([-130, -65, 24, 50], crs=ccrs.PlateCarree())
    
    # Gridlines
    gl = ax.gridlines(draw_labels=True, linewidth=0.5, 
                     color='#444444', alpha=0.5, linestyle='--')
    gl.top_labels = gl.right_labels = False
    gl.xlabel_style = {'size': 9, 'color': 'white'}
    gl.ylabel_style = {'size': 9, 'color': 'white'}
    
    ax.set_title('Dominant Development Type', fontsize=12, fontweight='bold', pad=8, color='white')
    
    # Colorbar
    cbar = plt.colorbar(sc, ax=ax, orientation='horizontal', 
                       pad=0.05, shrink=0.9, aspect=25, ticks=[0.375, 1.125, 1.875, 2.625])
    cbar.set_ticklabels(['Open\nSpace', 'Low\nIntensity', 'Medium\nIntensity', 'High\nIntensity'])
    cbar.ax.tick_params(labelsize=8, colors='white')
    cbar.outline.set_edgecolor('white')
    
    # Tight layout
    plt.subplots_adjust(hspace=0.15, wspace=0.1, left=0.05, right=0.95, top=0.98, bottom=0.02)
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='black')
        plt.savefig(save_path.replace('.png', '.pdf'), dpi=300, bbox_inches='tight', facecolor='black')
        print(f"Saved: {save_path}")
    
    return fig

# Run
fig = plot_urban_grid(reaches_nlcd,
    '/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/map_grid.png')
plt.show()

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

def plot_reaches_geometry(gdf, save_path=None):
    """Plot reaches with their actual geometry (not centroids)"""
    
    gdf = gdf.to_crs(epsg=4326).copy()
    
    # Load states
    try:
        states = gpd.read_file('/home/kwanhyuckkim_umass_edu/back_folder/tl_2012_us_state.shp')
        states = states.to_crs(epsg=4326).cx[-170:-50, 12:85]
        has_states = True
    except:
        has_states = False
    
    # Create figure
    fig, ax = plt.subplots(figsize=(18, 12), 
                          subplot_kw={'projection': ccrs.PlateCarree()},
                          facecolor='black')
    ax.set_facecolor('#1a1a1a')
    
    # Add features
    ax.add_feature(cfeature.LAND.with_scale('50m'), color='#2a2a2a')
    ax.add_feature(cfeature.COASTLINE.with_scale('50m'), linewidth=0.8, edgecolor='#666666')
    ax.add_feature(cfeature.BORDERS.with_scale('50m'), linewidth=0.8, edgecolor='#666666')
    ax.add_feature(cfeature.LAKES.with_scale('50m'), color='#1a3a52')
    
    # Add states
    if has_states:
        ax.add_geometries(states.geometry, ccrs.PlateCarree(), 
                         edgecolor='#555555', facecolor='none', linewidth=0.8)
    
    # Check geometry type
    geom_types = gdf.geometry.type.unique()
    print(f"Geometry types: {geom_types}")
    
    # Plot based on geometry type
    if 'LineString' in geom_types or 'MultiLineString' in geom_types:
        # Plot as lines
        for idx, row in gdf.iterrows():
            if row.geometry is not None:
                if row.geometry.geom_type in ['LineString', 'MultiLineString']:
                    ax.plot(*row.geometry.xy, color='cyan', linewidth=1.5, 
                           alpha=0.7, transform=ccrs.PlateCarree())
    elif 'Polygon' in geom_types or 'MultiPolygon' in geom_types:
        # Plot as polygons
        gdf.plot(ax=ax, facecolor='cyan', edgecolor='white', 
                linewidth=0.5, alpha=0.6, transform=ccrs.PlateCarree())
    else:
        # Plot as points
        gdf.plot(ax=ax, markersize=30, color='cyan', 
                edgecolor='white', linewidth=0.5, alpha=0.7)
    
    # Set extent
    ax.set_extent([-130, -65, 24, 50], crs=ccrs.PlateCarree())
    
    # Gridlines
    gl = ax.gridlines(draw_labels=True, linewidth=0.5, 
                     color='#444444', alpha=0.5, linestyle='--')
    gl.top_labels = gl.right_labels = False
    gl.xlabel_style = {'size': 12, 'color': 'white'}
    gl.ylabel_style = {'size': 12, 'color': 'white'}
    
    # Title
    ax.set_title(f'SWOT Reaches Geometry (n={len(gdf):,})', 
                fontsize=16, fontweight='bold', pad=15, color='white')
    
    # Add info box
    info = f"Total reaches: {len(gdf):,}\n"
    info += f"Geometry type: {', '.join(geom_types)}\n"
    info += f"Urban range: {gdf['fctimp_mean'].min():.1%} - {gdf['fctimp_mean'].max():.1%}"
    
    ax.text(0.02, 0.98, info, transform=ax.transAxes, fontsize=11,
           va='top', family='monospace', color='white',
           bbox=dict(boxstyle='round', facecolor='black', alpha=0.8, 
                    edgecolor='cyan', linewidth=2))
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='black')
        print(f"Saved: {save_path}")
    
    return fig, ax

# Plot
fig, ax = plot_reaches_geometry(reaches_nlcd,
    '/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/reaches_geometry.png')
plt.show()

In [ ]:
"""
Interactive map - US focused with dark background
"""
import plotly.graph_objects as go

def create_us_focused_interactive_map(gdf, save_path=None):
    """
    Create interactive map focused on US with dark theme
    """
    
    # Prepare data
    gdf = gdf.to_crs(epsg=4326).copy()
    gdf['centroid'] = gdf.geometry.centroid
    gdf['lon'] = gdf.centroid.x
    gdf['lat'] = gdf.centroid.y
    
    # Create custom hover text
    hover_text = []
    for _, row in gdf.iterrows():
        text = f"<b>Reach ID: {row['reach_id']}</b><br>"
        text += f"<b>USGS Site: {row['site_no']}</b><br><br>"
        text += f"Total Impervious: {row['fctimp_mean']:.3f}<br>"
        text += f"Open Space: {row['nlcd_21_mean']:.3f}<br>"
        text += f"Low Intensity: {row['nlcd_22_mean']:.3f}<br>"
        text += f"Medium Intensity: {row['nlcd_23_mean']:.3f}<br>"
        text += f"High Intensity: {row['nlcd_24_mean']:.3f}<br>"
        text += f"<br>Lat: {row['lat']:.3f}, Lon: {row['lon']:.3f}"
        hover_text.append(text)
    
    # Create figure
    fig = go.Figure()
    
    fig.add_trace(
        go.Scattermapbox(
            lat=gdf['lat'],
            lon=gdf['lon'],
            mode='markers',
            marker=dict(
                size=gdf['fctimp_mean'] * 100 + 8,
                color=gdf['fctimp_mean'],
                colorscale=[[0, 'white'], [1, 'red']],
                cmin=0,
                cmax=gdf['fctimp_mean'].max(),
                colorbar=dict(
                    title=dict(
                        text="Total<br>Impervious",
                        font=dict(size=14, color='white')
                    ),
                    tickformat='.3f',
                    thickness=20,
                    len=0.7,
                    x=1.02,
                    tickfont=dict(size=12, color='white')
                ),
                opacity=0.8
            ),
            text=hover_text,
            hovertemplate='%{text}<extra></extra>',
            name='SWOT Reaches'
        )
    )
    
    # Update layout
    fig.update_layout(
        title=dict(
            text=f'Interactive SWOT Reaches Map (n={len(gdf):,})',
            font=dict(size=18, color='white')
        ),
        mapbox=dict(
            style='carto-darkmatter',
            center={'lat': 39, 'lon': -98},
            zoom=3.5
        ),
        height=700,
        margin=dict(l=0, r=0, t=40, b=0),
        font=dict(size=12, color='white'),
        paper_bgcolor='#1a1a1a',
        plot_bgcolor='#1a1a1a'
    )
    
    if save_path:
        fig.write_html(save_path)
        print(f"Interactive map saved: {save_path}")
    
    return fig


def create_multi_layer_us_map(gdf, save_path=None):
    """
    Multi-layer version with US focus and dark theme
    """
    
    # Prepare data
    gdf = gdf.to_crs(epsg=4326).copy()
    gdf['centroid'] = gdf.geometry.centroid
    gdf['lon'] = gdf.centroid.x
    gdf['lat'] = gdf.centroid.y
    
    # Create figure
    fig = go.Figure()
    
    # Define layers
    layers = {
        'Total Impervious': ('fctimp_mean', [[0, 'white'], [1, 'red']]),
        'Open Space': ('nlcd_21_mean', 'Greens'),
        'Low Intensity': ('nlcd_22_mean', 'Blues'),
        'Medium Intensity': ('nlcd_23_mean', 'Oranges'),
        'High Intensity': ('nlcd_24_mean', 'Reds')
    }
    
    # Add traces for each layer
    for idx, (name, (col, colorscale)) in enumerate(layers.items()):
        
        # Custom hover text
        hover_text = []
        for _, row in gdf.iterrows():
            text = f"<b>Reach ID: {row['reach_id']}</b><br>"
            text += f"<b>USGS Site: {row['site_no']}</b><br><br>"
            text += f"<b>{name}: {row[col]:.3f}</b><br><br>"
            text += f"Total Impervious: {row['fctimp_mean']:.3f}<br>"
            text += f"Open Space: {row['nlcd_21_mean']:.3f}<br>"
            text += f"Low Intensity: {row['nlcd_22_mean']:.3f}<br>"
            text += f"Medium Intensity: {row['nlcd_23_mean']:.3f}<br>"
            text += f"High Intensity: {row['nlcd_24_mean']:.3f}<br>"
            text += f"<br>Lat: {row['lat']:.3f}, Lon: {row['lon']:.3f}"
            hover_text.append(text)
        
        fig.add_trace(
            go.Scattermapbox(
                lat=gdf['lat'],
                lon=gdf['lon'],
                mode='markers',
                marker=dict(
                    size=gdf[col] * 80 + 8,
                    color=gdf[col],
                    colorscale=colorscale,
                    cmin=0,
                    cmax=gdf[col].max(),
                    colorbar=dict(
                        title=dict(text=name, font=dict(size=14, color='white')),
                        tickformat='.3f',
                        thickness=20,
                        len=0.7,
                        x=1.02,
                        tickfont=dict(color='white')
                    ),
                    opacity=0.8
                ),
                text=hover_text,
                hovertemplate='%{text}<extra></extra>',
                name=name,
                visible=(idx == 0)
            )
        )
    
    # Create dropdown menu
    buttons = []
    for idx, (name, (col, colorscale)) in enumerate(layers.items()):
        visible = [False] * len(layers)
        visible[idx] = True
        buttons.append(
            dict(
                label=name,
                method='update',
                args=[
                    {'visible': visible},
                    {'title': f'{name} Distribution (n={len(gdf):,})'}
                ]
            )
        )
    
    # Update layout
    fig.update_layout(
        title=dict(
            text=f'Total Impervious Distribution (n={len(gdf):,})',
            font=dict(size=18, color='white')
        ),
        mapbox=dict(
            style='carto-darkmatter',
            center={'lat': 39, 'lon': -98},
            zoom=3.5
        ),
        updatemenus=[
            dict(
                buttons=buttons,
                direction='down',
                pad={'r': 10, 't': 10},
                showactive=True,
                x=0.01,
                xanchor='left',
                y=0.99,
                yanchor='top',
                bgcolor='rgba(50, 50, 50, 0.9)',
                bordercolor='white',
                borderwidth=2,
                font=dict(size=12, color='white')
            )
        ],
        height=700,
        margin=dict(l=0, r=0, t=50, b=0),
        font=dict(size=12, color='white'),
        paper_bgcolor='#1a1a1a'
    )
    
    if save_path:
        fig.write_html(save_path)
        print(f"Multi-layer interactive map saved: {save_path}")
    
    return fig


# Create both versions
print("Creating US-focused interactive maps with dark theme...")

# Simple version
fig1 = create_us_focused_interactive_map(
    reaches_nlcd,
    save_path='/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/interactive_map_us.html'
)
fig1.show()

# Multi-layer version
fig2 = create_multi_layer_us_map(
    reaches_nlcd,
    save_path='/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/interactive_map_us_layers.html'
)
fig2.show()

In [ ]:
"""
Select 12 reaches without overlap by starting from highest percentile
"""

import pandas as pd
import numpy as np

def select_12_reaches_no_overlap(gdf, column='fctimp_mean'):
    """
    Select 12 reaches based on percentiles without any overlap
    
    Strategy:
    1. Start from 100% (highest value)
    2. Find closest match
    3. Remove selected from pool
    4. Move to next lower percentile
    5. Repeat
    
    Parameters:
    -----------
    gdf : GeoDataFrame
        Data with reaches
    column : str
        Column to use for stratification
    
    Returns:
    --------
    selected_gdf : GeoDataFrame
        Selected 12 reaches (no overlap)
    """
    
    # Calculate target values
    max_val = gdf[column].max()
    percentages = np.linspace(1, 100, 12)  # [1, 10, 19, 28, ..., 100]
    targets = [max_val * p / 100 for p in percentages]
    
    # Start from highest to lowest (reverse order)
    percentages_reversed = percentages[::-1]  # [100, 91, 82, ...]
    targets_reversed = targets[::-1]
    
    # Track selected indices
    selected_indices = []
    available_pool = gdf.index.tolist()  # All indices available initially
    
    print("Selecting 12 Reaches (No Overlap)")
    
    for p, target in zip(percentages_reversed, targets_reversed):
        # Filter to only available reaches
        available_gdf = gdf.loc[available_pool]
        
        # Find closest match in available pool
        diff = (available_gdf[column] - target).abs()
        closest_idx = diff.idxmin()
        
        # Get the actual value
        actual_val = gdf.loc[closest_idx, column]
        
        print(f"\n{p:.0f}% target: {target:.4f}")
        print(f"  -> Selected: {actual_val:.4f} (diff={abs(actual_val-target):.4f})")
        print(f"     Reach ID: {gdf.loc[closest_idx, 'reach_id']}")
        print(f"     USGS Site: {gdf.loc[closest_idx, 'site_no']}")
        
        # Add to selected list
        selected_indices.append(closest_idx)
        
        # Remove from available pool
        available_pool.remove(closest_idx)
    
    # Get selected reaches (reverse back to ascending order)
    selected_indices_sorted = selected_indices[::-1]  # Back to 1% -> 100%
    selected_gdf = gdf.loc[selected_indices_sorted].copy()
    
    # Add metadata
    selected_gdf['percentile'] = percentages  # 1, 10, 19, ..., 100
    selected_gdf['site_number'] = range(1, 13)
    selected_gdf['target_value'] = targets
    
    print("Selection Complete - No Overlaps!")
    print(f"Selected {len(selected_gdf)} unique reaches")
    print(f"Urban range: {selected_gdf[column].min():.4f} - {selected_gdf[column].max():.4f}")
    
    return selected_gdf


# Execute selection
selected_12 = select_12_reaches_no_overlap(reaches_nlcd, column='fctimp_mean')

In [ ]:
selected_12_prev = gpd.read_file('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12.shp')
selected_12_prev.site_no

In [ ]:
selected_12.site_no

In [ ]:
selected_12.to_file('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12_vD.shp')

In [ ]:
"""
Step 2: Create summary table for selected 12 reaches
"""

def create_summary_table(selected_gdf):
    """
    Create formatted summary table for selected reaches
    
    Parameters:
    -----------
    selected_gdf : GeoDataFrame
        Selected reaches with percentile info
    
    Returns:
    --------
    summary_df : DataFrame
        Formatted summary table
    """
    
    # Get centroid coordinates
    selected_gdf['centroid'] = selected_gdf.geometry.centroid
    selected_gdf['lon'] = selected_gdf.centroid.x
    selected_gdf['lat'] = selected_gdf.centroid.y
    
    # Extract key information
    summary = selected_gdf[[
        'site_number', 'percentile', 'reach_id', 'site_no',
        'fctimp_mean', 'nlcd_21_mean', 'nlcd_22_mean', 
        'nlcd_23_mean', 'nlcd_24_mean', 'lat', 'lon'
    ]].copy()
    
    # Determine dominant development type
    urban_cols = ['nlcd_21_mean', 'nlcd_22_mean', 'nlcd_23_mean', 'nlcd_24_mean']
    summary['dominant_idx'] = summary[urban_cols].idxmax(axis=1)
    
    # Map to category names
    dominant_map = {
        'nlcd_21_mean': 'Open Space',
        'nlcd_22_mean': 'Low Intensity',
        'nlcd_23_mean': 'Medium Intensity',
        'nlcd_24_mean': 'High Intensity'
    }
    summary['Dominant Development'] = summary['dominant_idx'].map(dominant_map)
    summary = summary.drop(columns=['dominant_idx'])
    
    # Rename columns for clarity
    summary = summary.rename(columns={
        'site_number': 'Site',
        'percentile': 'Percentile',
        'reach_id': 'SWORD Reach ID',
        'site_no': 'USGS Site',
        'fctimp_mean': 'Total Impervious',
        'nlcd_21_mean': 'Open Space',
        'nlcd_22_mean': 'Low Intensity',
        'nlcd_23_mean': 'Medium Intensity',
        'nlcd_24_mean': 'High Intensity',
        'lat': 'Latitude',
        'lon': 'Longitude'
    })
    
    # Reorder columns
    summary = summary[[
        'Site', 'Percentile', 'Dominant Development', 'SWORD Reach ID', 'USGS Site',
        'Total Impervious', 'Open Space', 'Low Intensity', 'Medium Intensity', 'High Intensity',
        'Latitude', 'Longitude'
    ]]
    
    return summary


# Create summary table
print("Summary Table of Selected 12 Reaches")

summary_table = create_summary_table(selected_12)

# Display with formatting
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.precision', 3)

print("\n", summary_table.to_string(index=False))

# Save to CSV
output_csv = '/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12_reaches.csv'
summary_table.to_csv(output_csv, index=False, float_format='%.4f')
print(f"\nSaved: {output_csv}")

# Dominant Development breakdown
print("Dominant Development Breakdown:")
dev_counts = summary_table['Dominant Development'].value_counts()
for dev_type, count in dev_counts.items():
    print(f"  {dev_type:20s}: {count:2d} sites")

In [ ]:
"""
Interactive map for selected 12 reaches only - Multi-layer version
"""
import plotly.graph_objects as go

def create_selected_12_multilayer_map(selected_gdf, save_path=None):
    """
    Create multi-layer interactive map for selected 12 reaches only
    
    Parameters:
    -----------
    selected_gdf : GeoDataFrame
        Selected 12 reaches
    save_path : str, optional
        Path to save HTML file
    
    Returns:
    --------
    fig : plotly Figure
    """
    
    # Prepare data
    gdf = selected_gdf.to_crs(epsg=4326).copy()
    gdf['centroid'] = gdf.geometry.centroid
    gdf['lon'] = gdf.centroid.x
    gdf['lat'] = gdf.centroid.y
    
    # Sort by site number for consistency
    gdf = gdf.sort_values('site_number').reset_index(drop=True)
    
    # Create figure
    fig = go.Figure()
    
    # Define layers
    layers = {
        'Total Impervious': ('fctimp_mean', [[0, 'white'], [1, 'red']]),
        'Open Space': ('nlcd_21_mean', 'Greens'),
        'Low Intensity': ('nlcd_22_mean', 'Blues'),
        'Medium Intensity': ('nlcd_23_mean', 'Oranges'),
        'High Intensity': ('nlcd_24_mean', 'Reds')
    }
    
    # Add traces for each layer
    for idx, (name, (col, colorscale)) in enumerate(layers.items()):
        
        # Custom hover text
        hover_text = []
        for _, row in gdf.iterrows():
            text = f"<b>Site #{row['site_number']} ({row['percentile']:.0f}th percentile)</b><br><br>"
            text += f"<b>Reach ID: {row['reach_id']}</b><br>"
            text += f"<b>USGS Site: {row['site_no']}</b><br><br>"
            text += f"<b>{name}: {row[col]:.3f}</b><br><br>"
            text += f"Total Impervious: {row['fctimp_mean']:.3f}<br>"
            text += f"Open Space: {row['nlcd_21_mean']:.3f}<br>"
            text += f"Low Intensity: {row['nlcd_22_mean']:.3f}<br>"
            text += f"Medium Intensity: {row['nlcd_23_mean']:.3f}<br>"
            text += f"High Intensity: {row['nlcd_24_mean']:.3f}<br>"
            text += f"<br>Lat: {row['lat']:.3f}, Lon: {row['lon']:.3f}"
            hover_text.append(text)
        
        fig.add_trace(
            go.Scattermapbox(
                lat=gdf['lat'],
                lon=gdf['lon'],
                mode='markers+text',
                marker=dict(
                    size=gdf[col] * 40 + 10,  # Larger since only 12 points
                    color=gdf[col],
                    colorscale=colorscale,
                    cmin=0,
                    cmax=gdf[col].max(),
                    colorbar=dict(
                        title=dict(text=name, font=dict(size=14, color='white')),
                        tickformat='.3f',
                        thickness=20,
                        len=0.7,
                        x=1.02,
                        tickfont=dict(color='white')
                    ),
                    opacity=0.8
                ),
                text=gdf['site_number'].astype(str),
                textposition='middle center',
                textfont=dict(size=12, color='white', family='Arial Black'),
                hovertext=hover_text,
                hovertemplate='%{hovertext}<extra></extra>',
                name=name,
                visible=(idx == 0)
            )
        )
    
    # Create dropdown menu
    buttons = []
    for idx, (name, (col, colorscale)) in enumerate(layers.items()):
        visible = [False] * len(layers)
        visible[idx] = True
        buttons.append(
            dict(
                label=name,
                method='update',
                args=[
                    {'visible': visible},
                    {'title': f'{name} - Selected 12 Reaches'}
                ]
            )
        )
    
    # Update layout
    fig.update_layout(
        title=dict(
            text=f'Selected 12 Reaches - Total Impervious',
            font=dict(size=18, color='white')
        ),
        mapbox=dict(
            style='carto-darkmatter',
            center={'lat': 39, 'lon': -98},
            zoom=3.5
        ),
        updatemenus=[
            dict(
                buttons=buttons,
                direction='down',
                pad={'r': 10, 't': 10},
                showactive=True,
                x=0.01,
                xanchor='left',
                y=0.99,
                yanchor='top',
                bgcolor='rgba(50, 50, 50, 0.9)',
                bordercolor='white',
                borderwidth=2,
                font=dict(size=12, color='white')
            )
        ],
        height=800,
        margin=dict(l=0, r=0, t=50, b=0),
        font=dict(size=12, color='white'),
        paper_bgcolor='#1a1a1a'
    )
    
    if save_path:
        fig.write_html(save_path)
        print(f"Multi-layer map (12 reaches) saved: {save_path}")
    
    return fig


# Create multi-layer map for selected 12 reaches
print("Creating Multi-Layer Interactive Map (Selected 12 Reaches)")

fig_12_multilayer = create_selected_12_multilayer_map(
    selected_12,
    save_path='/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12_multilayer.html'
)

# Display
fig_12_multilayer

In [ ]:
"""
Interactive map with SWORD reaches + Buffer polygons + USGS gages (snapped)
"""
import geopandas as gpd
import plotly.graph_objects as go
import json
import pandas as pd
from shapely.geometry import Point
from shapely.ops import nearest_points

def snap_point_to_line(point, line):
    """
    Snap a point to the nearest location on a line
    
    Parameters:
    -----------
    point : shapely Point
        Point to snap
    line : shapely LineString or MultiLineString
        Line to snap to
    
    Returns:
    --------
    snapped_point : shapely Point
        Point snapped to line
    """
    # Project point onto line
    distance_along_line = line.project(point)
    snapped_point = line.interpolate(distance_along_line)
    return snapped_point


def create_sword_buffer_gage_map(selected_gdf, sword_shp_path, usgs_csv_path, save_path=None):
    """
    Create interactive map showing:
    1. SWORD reaches (linestrings) - bright green background
    2. Buffer polygons colored by total impervious - interactive
    3. USGS gages snapped to SWORD reaches - small points
    4. Dropdown menu to zoom into each site
    
    Parameters:
    -----------
    selected_gdf : GeoDataFrame
        Selected 12 reaches with buffer polygons
    sword_shp_path : str
        Path to SWORD reaches shapefile
    usgs_csv_path : str
        Path to USGS locations CSV
    save_path : str, optional
        Path to save HTML
    
    Returns:
    --------
    fig : plotly Figure
    """
    
    print("Loading USGS gage locations...")
    # Load USGS gage locations
    usgs_loc = pd.read_csv(usgs_csv_path, dtype={'gage_id': object})
    usgs_loc = usgs_loc.rename(columns={'gage_id': 'site_no', \
                                        'gage_latitude': 'dec_lat_va',\
                                        'gage_longitude': 'dec_long_va',\
                                        'gage_name': 'station_nm'})
    print(f"Loaded {len(usgs_loc)} USGS gages")
    
    print("Loading SWORD reaches...")
    # Load SWORD reaches
    sword_reaches = gpd.read_file(sword_shp_path)
    sword_reaches = sword_reaches.to_crs(epsg=4326)
    
    # Filter to only selected reach_ids
    selected_reach_ids = selected_gdf['reach_id'].tolist()
    sword_selected = sword_reaches[sword_reaches['reach_id'].isin(selected_reach_ids)].copy()
    
    print(f"Loaded {len(sword_selected)} SWORD reaches (linestrings)")
    
    # Prepare selected buffers
    selected_gdf = selected_gdf.to_crs(epsg=4326).copy()
    selected_gdf = selected_gdf.sort_values('site_number').reset_index(drop=True)
    
    print(f"Loaded {len(selected_gdf)} buffer polygons")
    
    # Match USGS gages to selected reaches and snap to SWORD lines
    print("Snapping USGS gages to SWORD reaches...")
    snapped_gages = []
    
    for _, row in selected_gdf.iterrows():
        site_no = row['site_no']
        reach_id = row['reach_id']
        
        # Find USGS gage location
        usgs_match = usgs_loc[usgs_loc['site_no'] == site_no]
        
        if len(usgs_match) > 0:
            gage_lat = usgs_match.iloc[0]['dec_lat_va']
            gage_lon = usgs_match.iloc[0]['dec_long_va']
            gage_name = usgs_match.iloc[0]['station_nm']
            
            # Create point
            gage_point = Point(gage_lon, gage_lat)
            
            # Find corresponding SWORD reach
            sword_match = sword_selected[sword_selected['reach_id'] == reach_id]
            
            if len(sword_match) > 0:
                sword_geom = sword_match.iloc[0].geometry
                
                # Snap point to line
                snapped_point = snap_point_to_line(gage_point, sword_geom)
                
                snapped_gages.append({
                    'site_no': site_no,
                    'reach_id': reach_id,
                    'site_number': row['site_number'],
                    'station_nm': gage_name,
                    'original_lat': gage_lat,
                    'original_lon': gage_lon,
                    'snapped_lat': snapped_point.y,
                    'snapped_lon': snapped_point.x
                })
    
    snapped_gages_df = pd.DataFrame(snapped_gages)
    print(f"Snapped {len(snapped_gages_df)} USGS gages to SWORD reaches")
    
    # Calculate bounds for each site (for zoom functionality)
    zoom_info = []
    for idx, row in selected_gdf.iterrows():
        bounds = row.geometry.bounds  # (minx, miny, maxx, maxy)
        center_lon = (bounds[0] + bounds[2]) / 2
        center_lat = (bounds[1] + bounds[3]) / 2
        
        # Calculate zoom level based on bounds size
        lon_range = bounds[2] - bounds[0]
        lat_range = bounds[3] - bounds[1]
        max_range = max(lon_range, lat_range)
        
        if max_range > 1:
            zoom = 8
        elif max_range > 0.5:
            zoom = 9
        elif max_range > 0.2:
            zoom = 10
        elif max_range > 0.1:
            zoom = 11
        else:
            zoom = 12
        
        zoom_info.append({
            'site_number': row['site_number'],
            'center_lat': center_lat,
            'center_lon': center_lon,
            'zoom': zoom,
            'reach_id': row['reach_id']
        })
    
    # Create figure
    fig = go.Figure()
    
    # Layer 1: SWORD reaches (linestrings) - bright green
    for idx, row in sword_selected.iterrows():
        geom = row.geometry
        
        if geom.geom_type == 'LineString':
            lons, lats = geom.xy
            fig.add_trace(
                go.Scattermapbox(
                    lon=list(lons),
                    lat=list(lats),
                    mode='lines',
                    line=dict(width=4, color='#00ff00'),  # Bright green
                    opacity=0.8,
                    hoverinfo='skip',
                    showlegend=False
                )
            )
        elif geom.geom_type == 'MultiLineString':
            for line in geom.geoms:
                lons, lats = line.xy
                fig.add_trace(
                    go.Scattermapbox(
                        lon=list(lons),
                        lat=list(lats),
                        mode='lines',
                        line=dict(width=4, color='#00ff00'),  # Bright green
                        opacity=0.8,
                        hoverinfo='skip',
                        showlegend=False
                    )
                )
    
    # Add legend entry for SWORD reaches
    fig.add_trace(
        go.Scattermapbox(
            lon=[None],
            lat=[None],
            mode='lines',
            line=dict(width=4, color='#00ff00'),
            name='SWORD Reaches',
            showlegend=True
        )
    )
    
    # Layer 2: Buffer polygons with choropleth (interactive)
    # Create clean GeoDataFrame for GeoJSON conversion
    gdf_for_json = selected_gdf[['geometry', 'fctimp_mean', 'site_number', 
                                  'reach_id', 'site_no', 'percentile',
                                  'nlcd_21_mean', 'nlcd_22_mean', 
                                  'nlcd_23_mean', 'nlcd_24_mean']].copy()
    gdf_for_json['id'] = gdf_for_json.index
    
    # Calculate centroids for lat/lon display
    centroids = selected_gdf.geometry.centroid
    lons = centroids.x.values
    lats = centroids.y.values
    
    # Prepare hover text
    hover_text = []
    for idx, row in gdf_for_json.iterrows():
        text = f"<b>Site #{row['site_number']} ({row['percentile']:.0f}th percentile)</b><br><br>"
        text += f"<b>Reach ID: {row['reach_id']}</b><br>"
        text += f"<b>USGS Site: {row['site_no']}</b><br><br>"
        text += f"<b>Total Impervious: {row['fctimp_mean']:.3f}</b><br><br>"
        text += f"Total Impervious: {row['fctimp_mean']:.3f}<br>"
        text += f"Open Space: {row['nlcd_21_mean']:.3f}<br>"
        text += f"Low Intensity: {row['nlcd_22_mean']:.3f}<br>"
        text += f"Medium Intensity: {row['nlcd_23_mean']:.3f}<br>"
        text += f"High Intensity: {row['nlcd_24_mean']:.3f}<br><br>"
        text += f"Lat: {lats[idx]:.3f}, Lon: {lons[idx]:.3f}"
        hover_text.append(text)
    
    # Convert to GeoJSON
    gdf_clean = gdf_for_json[['geometry', 'fctimp_mean', 'id']].copy()
    geojson_clean = json.loads(gdf_clean.to_json())
    
    # Add choropleth for polygons
    fig.add_trace(
        go.Choroplethmapbox(
            geojson=geojson_clean,
            locations=gdf_clean['id'],
            z=gdf_clean['fctimp_mean'],
            colorscale=[[0, 'white'], [1, 'red']],
            zmin=0,
            zmax=gdf_clean['fctimp_mean'].max(),
            marker=dict(opacity=0.7, line=dict(width=2, color='white')),
            colorbar=dict(
                title=dict(text="Total<br>Impervious", font=dict(size=14, color='white')),
                tickformat='.3f',
                thickness=20,
                len=0.7,
                x=1.02,
                tickfont=dict(color='white')
            ),
            hovertemplate='%{customdata}<extra></extra>',
            customdata=hover_text,
            name='Buffer Polygons',
            showlegend=True
        )
    )
    
    # Layer 3: USGS gages (snapped to SWORD reaches) - small points
    if len(snapped_gages_df) > 0:
        # Prepare hover text for gages
        gage_hover = []
        for _, row in snapped_gages_df.iterrows():
            text = f"<b>USGS Gage</b><br>"
            text += f"Site #{row['site_number']}<br>"
            text += f"Site No: {row['site_no']}<br>"
            text += f"Station: {row['station_nm']}<br><br>"
            text += f"Original: {row['original_lat']:.4f}, {row['original_lon']:.4f}<br>"
            text += f"Snapped: {row['snapped_lat']:.4f}, {row['snapped_lon']:.4f}"
            gage_hover.append(text)
        
        fig.add_trace(
            go.Scattermapbox(
                lon=snapped_gages_df['snapped_lon'],
                lat=snapped_gages_df['snapped_lat'],
                mode='markers',
                marker=dict(
                    size=10,
                    color='red',
                    symbol='circle',
                    opacity=1.0
                ),
                text=gage_hover,
                hovertemplate='%{text}<extra></extra>',
                name='USGS Gages',
                showlegend=True
            )
        )
    
    # Create dropdown buttons for zoom
    zoom_buttons = []
    
    # Add "All Sites" button
    zoom_buttons.append(
        dict(
            label='All Sites (US View)',
            method='relayout',
            args=[{
                'mapbox.center': {'lat': 39, 'lon': -98},
                'mapbox.zoom': 3.5
            }]
        )
    )
    
    # Add button for each site
    for info in zoom_info:
        zoom_buttons.append(
            dict(
                label=f'Site #{info["site_number"]} - {info["reach_id"]}',
                method='relayout',
                args=[{
                    'mapbox.center': {'lat': info['center_lat'], 'lon': info['center_lon']},
                    'mapbox.zoom': info['zoom']
                }]
            )
        )
    
    # Update layout
    fig.update_layout(
        title=dict(
            text='Selected 12 Reaches: SWORD + Buffers + USGS Gages',
            font=dict(size=18, color='white')
        ),
        mapbox=dict(
            style='carto-darkmatter',
            center={'lat': 39, 'lon': -98},
            zoom=3.5
        ),
        updatemenus=[
            dict(
                buttons=zoom_buttons,
                direction='down',
                pad={'r': 10, 't': 10},
                showactive=True,
                x=0.01,
                xanchor='left',
                y=0.80,
                yanchor='top',
                bgcolor='rgba(50, 50, 50, 0.95)',
                bordercolor='cyan',
                borderwidth=2,
                font=dict(size=11, color='white')
            )
        ],
        height=800,
        margin=dict(l=0, r=0, t=50, b=0),
        font=dict(size=12, color='white'),
        paper_bgcolor='#1a1a1a',
        showlegend=True,
        legend=dict(
            x=0.01,
            y=0.99,
            bgcolor='rgba(50, 50, 50, 0.95)',
            bordercolor='white',
            borderwidth=2,
            font=dict(color='white', size=11)
        )
    )
    
    if save_path:
        fig.write_html(save_path)
        print(f"\nInteractive map saved: {save_path}")
    
    return fig, snapped_gages_df


# Create the map
print("Creating Interactive Map with USGS Gages")

fig_complete, snapped_gages = create_sword_buffer_gage_map(
    selected_12,
    sword_shp_path='/work/pi_kandread_umass_edu/swot-urban/1_data_processing/reaches_v1.shp',
    usgs_csv_path="/work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/paired_reach_SWOT_gage.csv",
    save_path='/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/complete_map.html'
)

print("Map Layers:")
print("  - SWORD Reaches: Bright green lines")
print("  - Buffer Polygons: White->Red (Total Impervious)")
print("  - USGS Gages: Small red points (snapped to reaches)")
print("  - Dropdown: Zoom to any site")

# Display snapped gages info
print("Snapped USGS Gages Summary")
print(snapped_gages[['site_number', 'site_no', 'station_nm', 'snapped_lat', 'snapped_lon']].to_string(index=False))

# Display
fig_complete

In [ ]:
"""
Map with USGS Satellite Imagery (No token needed!)
"""
import geopandas as gpd
import plotly.graph_objects as go
import json
import pandas as pd
from shapely.geometry import Point

def snap_point_to_line(point, line):
    """Snap a point to the nearest location on a line"""
    distance_along_line = line.project(point)
    snapped_point = line.interpolate(distance_along_line)
    return snapped_point


def create_sword_buffer_gage_map_imagery(selected_gdf, sword_shp_path, usgs_csv_path, save_path=None):
    """
    Create map with USGS satellite imagery background (no token needed!)
    """
    
    print("Loading data...")
    # Load USGS gage locations
    usgs_loc = pd.read_csv(usgs_csv_path, dtype={'gage_id': object})
    usgs_loc = usgs_loc.rename(columns={'gage_id': 'site_no', \
                                        'gage_latitude': 'dec_lat_va',\
                                        'gage_longitude': 'dec_long_va',\
                                        'gage_name': 'station_nm'})
    
    # Load SWORD reaches
    sword_reaches = gpd.read_file(sword_shp_path).to_crs(epsg=4326)
    selected_reach_ids = selected_gdf['reach_id'].tolist()
    sword_selected = sword_reaches[sword_reaches['reach_id'].isin(selected_reach_ids)].copy()
    
    # Prepare buffers
    selected_gdf = selected_gdf.to_crs(epsg=4326).copy().sort_values('site_number').reset_index(drop=True)
    
    # Snap USGS gages
    print("Snapping USGS gages...")
    snapped_gages = []
    for _, row in selected_gdf.iterrows():
        usgs_match = usgs_loc[usgs_loc['site_no'] == row['site_no']]
        if len(usgs_match) > 0:
            gage_point = Point(usgs_match.iloc[0]['dec_long_va'], usgs_match.iloc[0]['dec_lat_va'])
            sword_match = sword_selected[sword_selected['reach_id'] == row['reach_id']]
            if len(sword_match) > 0:
                snapped_point = snap_point_to_line(gage_point, sword_match.iloc[0].geometry)
                snapped_gages.append({
                    'site_no': row['site_no'],
                    'site_number': row['site_number'],
                    'station_nm': usgs_match.iloc[0]['station_nm'],
                    'snapped_lat': snapped_point.y,
                    'snapped_lon': snapped_point.x
                })
    
    snapped_gages_df = pd.DataFrame(snapped_gages)
    
    # Calculate zoom info
    zoom_info = []
    for _, row in selected_gdf.iterrows():
        bounds = row.geometry.bounds
        center_lon = (bounds[0] + bounds[2]) / 2
        center_lat = (bounds[1] + bounds[3]) / 2
        max_range = max(bounds[2] - bounds[0], bounds[3] - bounds[1])
        
        zoom = 12 if max_range < 0.1 else (11 if max_range < 0.2 else (10 if max_range < 0.5 else (9 if max_range < 1 else 8)))
        
        zoom_info.append({
            'site_number': row['site_number'],
            'center_lat': center_lat,
            'center_lon': center_lon,
            'zoom': zoom,
            'reach_id': row['reach_id']
        })
    
    # Create figure
    fig = go.Figure()
    
    # SWORD reaches - cyan color for contrast on satellite imagery
    for _, row in sword_selected.iterrows():
        geom = row.geometry
        if geom.geom_type == 'LineString':
            lons, lats = geom.xy
            fig.add_trace(go.Scattermapbox(
                lon=list(lons), lat=list(lats), mode='lines',
                line=dict(width=4, color='cyan'),
                opacity=0.9, hoverinfo='skip', showlegend=False
            ))
        elif geom.geom_type == 'MultiLineString':
            for line in geom.geoms:
                lons, lats = line.xy
                fig.add_trace(go.Scattermapbox(
                    lon=list(lons), lat=list(lats), mode='lines',
                    line=dict(width=4, color='cyan'),
                    opacity=0.9, hoverinfo='skip', showlegend=False
                ))
    
    # Legend for SWORD reaches
    fig.add_trace(go.Scattermapbox(
        lon=[None], lat=[None], mode='lines',
        line=dict(width=4, color='cyan'),
        name='SWORD Reaches', showlegend=True
    ))
    
    # Buffer polygons
    gdf_json = selected_gdf[['geometry', 'fctimp_mean', 'site_number', 'reach_id', 
                             'site_no', 'percentile', 'nlcd_21_mean', 'nlcd_22_mean',
                             'nlcd_23_mean', 'nlcd_24_mean']].copy()
    gdf_json['id'] = gdf_json.index
    
    centroids = selected_gdf.geometry.centroid
    lons, lats = centroids.x.values, centroids.y.values
    
    hover_text = []
    for idx, row in gdf_json.iterrows():
        text = f"<b>Site #{row['site_number']} ({row['percentile']:.0f}th percentile)</b><br><br>"
        text += f"<b>Reach ID: {row['reach_id']}</b><br>"
        text += f"<b>USGS Site: {row['site_no']}</b><br><br>"
        text += f"<b>Total Impervious: {row['fctimp_mean']:.3f}</b><br><br>"
        text += f"Open Space: {row['nlcd_21_mean']:.3f}<br>"
        text += f"Low Intensity: {row['nlcd_22_mean']:.3f}<br>"
        text += f"Medium Intensity: {row['nlcd_23_mean']:.3f}<br>"
        text += f"High Intensity: {row['nlcd_24_mean']:.3f}<br><br>"
        text += f"Lat: {lats[idx]:.3f}, Lon: {lons[idx]:.3f}"
        hover_text.append(text)
    
    gdf_clean = gdf_json[['geometry', 'fctimp_mean', 'id']].copy()
    geojson_clean = json.loads(gdf_clean.to_json())
    
    fig.add_trace(go.Choroplethmapbox(
        geojson=geojson_clean,
        locations=gdf_clean['id'],
        z=gdf_clean['fctimp_mean'],
        colorscale=[[0, 'rgba(255,255,255,0.3)'], [1, 'rgba(255,0,0,0.7)']],
        zmin=0, zmax=gdf_clean['fctimp_mean'].max(),
        marker=dict(opacity=0.6, line=dict(width=2, color='yellow')),
        colorbar=dict(
            title="Total<br>Impervious",
            tickformat='.3f'
        ),
        hovertemplate='%{customdata}<extra></extra>',
        customdata=hover_text,
        name='Buffer Polygons',
        showlegend=True
    ))
    
    # USGS gages
    if len(snapped_gages_df) > 0:
        gage_hover = []
        for _, row in snapped_gages_df.iterrows():
            text = f"<b>USGS Gage</b><br>"
            text += f"Site #{row['site_number']}<br>"
            text += f"Site No: {row['site_no']}<br>"
            text += f"Station: {row['station_nm']}"
            gage_hover.append(text)
        
        fig.add_trace(go.Scattermapbox(
            lon=snapped_gages_df['snapped_lon'],
            lat=snapped_gages_df['snapped_lat'],
            mode='markers',
            marker=dict(size=12, color='red', symbol='circle', opacity=1.0),
            text=gage_hover,
            hovertemplate='%{text}<extra></extra>',
            name='USGS Gages',
            showlegend=True
        ))
    
    # Dropdown buttons
    zoom_buttons = [dict(
        label='All Sites (US View)',
        method='relayout',
        args=[{'mapbox.center': {'lat': 39, 'lon': -98}, 'mapbox.zoom': 3.5}]
    )]
    
    for info in zoom_info:
        zoom_buttons.append(dict(
            label=f'Site #{info["site_number"]} - {info["reach_id"]}',
            method='relayout',
            args=[{'mapbox.center': {'lat': info['center_lat'], 'lon': info['center_lon']},
                   'mapbox.zoom': info['zoom']}]
        ))
    
    # KEY PART: USGS Imagery layer!
    fig.update_layout(
        mapbox=dict(
            style="white-bg",  # Empty base
            layers=[{
                "below": 'traces',  # Put imagery BELOW data
                "sourcetype": "raster",
                "sourceattribution": "United States Geological Survey",
                "source": [
                    "https://basemap.nationalmap.gov/arcgis/rest/services/USGSImageryOnly/MapServer/tile/{z}/{y}/{x}"
                ]
            }],
            center={'lat': 39, 'lon': -98},
            zoom=3.5
        ),
        updatemenus=[dict(
            buttons=zoom_buttons,
            direction='down',
            x=0.01, y=0.80,
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='cyan',
            borderwidth=2
        )],
        title='Selected 12 Reaches - USGS Satellite Imagery',
        height=800,
        margin=dict(l=0, r=0, t=50, b=0),
        showlegend=True,
        legend=dict(
            x=0.01, y=0.99,
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=2
        )
    )
    
    if save_path:
        fig.write_html(save_path)
        print(f"Saved: {save_path}")
    
    return fig, snapped_gages_df


# Create map with USGS satellite imagery
print("Creating Map with USGS Satellite Imagery (No Token Needed!)")

fig_imagery, snapped_gages = create_sword_buffer_gage_map_imagery(
    selected_12,
    sword_shp_path='/work/pi_kandread_umass_edu/swot-urban/1_data_processing/reaches_v1.shp',
    usgs_csv_path="/work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/paired_reach_SWOT_gage.csv",
    save_path='/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/map_usgs_imagery.html'
)

print("\nCreated map with USGS satellite imagery!")
print("  This shows actual satellite/aerial photos from USGS")
print("  No token required!")

fig_imagery

In [ ]:
def create_sword_buffer_gage_map_imagery(selected_gdf, sword_shp_path, usgs_csv_path, 
                                          swot_node_path=None, save_path=None):
    """
    Create map with USGS satellite imagery background (no token needed!)
    Now includes SWOT node locations!
    """
    
    print("Loading data...")
    # Load USGS gage locations
    usgs_loc = pd.read_csv(usgs_csv_path, dtype={'gage_id': object})
    usgs_loc = usgs_loc.rename(columns={'gage_id': 'site_no', 
                                        'gage_latitude': 'dec_lat_va',
                                        'gage_longitude': 'dec_long_va',
                                        'gage_name': 'station_nm'})
    
    # Load SWORD reaches
    sword_reaches = gpd.read_file(sword_shp_path).to_crs(epsg=4326)
    selected_reach_ids = selected_gdf['reach_id'].tolist()
    sword_selected = sword_reaches[sword_reaches['reach_id'].isin(selected_reach_ids)].copy()
    
    # Load SWOT node data if provided
    nodes_df = None
    if swot_node_path:
        print("Loading SWOT node data...")
        df_swot = pd.read_parquet(swot_node_path)
        # Get unique nodes for selected reaches
        nodes_df = df_swot[df_swot['reach_id'].isin(selected_reach_ids)].copy()
        # Get unique node locations (assuming columns like node_id, latitude, longitude or p_lat, p_lon)
        # Adjust column names based on your actual data structure
        if 'node_id' in nodes_df.columns:
            nodes_df = nodes_df.drop_duplicates(subset=['node_id'])[['node_id', 'reach_id', 'lat', 'lon']].copy()
            print(f"  Found {len(nodes_df)} unique nodes")
    
    # Prepare buffers
    selected_gdf = selected_gdf.to_crs(epsg=4326).copy().sort_values('site_number').reset_index(drop=True)
    
    # Snap USGS gages
    print("Snapping USGS gages...")
    snapped_gages = []
    for _, row in selected_gdf.iterrows():
        usgs_match = usgs_loc[usgs_loc['site_no'] == row['site_no']]
        if len(usgs_match) > 0:
            gage_point = Point(usgs_match.iloc[0]['dec_long_va'], usgs_match.iloc[0]['dec_lat_va'])
            sword_match = sword_selected[sword_selected['reach_id'] == row['reach_id']]
            if len(sword_match) > 0:
                snapped_point = snap_point_to_line(gage_point, sword_match.iloc[0].geometry)
                snapped_gages.append({
                    'site_no': row['site_no'],
                    'site_number': row['site_number'],
                    'station_nm': usgs_match.iloc[0]['station_nm'],
                    'snapped_lat': snapped_point.y,
                    'snapped_lon': snapped_point.x
                })
    
    snapped_gages_df = pd.DataFrame(snapped_gages)
    
    # Calculate zoom info
    zoom_info = []
    for _, row in selected_gdf.iterrows():
        bounds = row.geometry.bounds
        center_lon = (bounds[0] + bounds[2]) / 2
        center_lat = (bounds[1] + bounds[3]) / 2
        max_range = max(bounds[2] - bounds[0], bounds[3] - bounds[1])
        
        zoom = 12 if max_range < 0.1 else (11 if max_range < 0.2 else (10 if max_range < 0.5 else (9 if max_range < 1 else 8)))
        
        zoom_info.append({
            'site_number': row['site_number'],
            'center_lat': center_lat,
            'center_lon': center_lon,
            'zoom': zoom,
            'reach_id': row['reach_id']
        })
    
    # Create figure
    fig = go.Figure()
    
    # SWORD reaches - cyan color for contrast on satellite imagery
    for _, row in sword_selected.iterrows():
        geom = row.geometry
        if geom.geom_type == 'LineString':
            lons, lats = geom.xy
            fig.add_trace(go.Scattermapbox(
                lon=list(lons), lat=list(lats), mode='lines',
                line=dict(width=4, color='cyan'),
                opacity=0.9, hoverinfo='skip', showlegend=False
            ))
        elif geom.geom_type == 'MultiLineString':
            for line in geom.geoms:
                lons, lats = line.xy
                fig.add_trace(go.Scattermapbox(
                    lon=list(lons), lat=list(lats), mode='lines',
                    line=dict(width=4, color='cyan'),
                    opacity=0.9, hoverinfo='skip', showlegend=False
                ))
    
    # Legend for SWORD reaches
    fig.add_trace(go.Scattermapbox(
        lon=[None], lat=[None], mode='lines',
        line=dict(width=4, color='cyan'),
        name='SWORD Reaches', showlegend=True
    ))
    
    # Buffer polygons
    gdf_json = selected_gdf[['geometry', 'fctimp_mean', 'site_number', 'reach_id', 
                             'site_no', 'percentile', 'nlcd_21_mean', 'nlcd_22_mean',
                             'nlcd_23_mean', 'nlcd_24_mean']].copy()
    gdf_json['id'] = gdf_json.index
    
    centroids = selected_gdf.geometry.centroid
    lons, lats = centroids.x.values, centroids.y.values
    
    hover_text = []
    for idx, row in gdf_json.iterrows():
        text = f"<b>Site #{row['site_number']} ({row['percentile']:.0f}th percentile)</b><br><br>"
        text += f"<b>Reach ID: {row['reach_id']}</b><br>"
        text += f"<b>USGS Site: {row['site_no']}</b><br><br>"
        text += f"<b>Total Impervious: {row['fctimp_mean']:.3f}</b><br><br>"
        text += f"Open Space: {row['nlcd_21_mean']:.3f}<br>"
        text += f"Low Intensity: {row['nlcd_22_mean']:.3f}<br>"
        text += f"Medium Intensity: {row['nlcd_23_mean']:.3f}<br>"
        text += f"High Intensity: {row['nlcd_24_mean']:.3f}<br><br>"
        text += f"Lat: {lats[idx]:.3f}, Lon: {lons[idx]:.3f}"
        hover_text.append(text)
    
    gdf_clean = gdf_json[['geometry', 'fctimp_mean', 'id']].copy()
    geojson_clean = json.loads(gdf_clean.to_json())
    
    fig.add_trace(go.Choroplethmapbox(
        geojson=geojson_clean,
        locations=gdf_clean['id'],
        z=gdf_clean['fctimp_mean'],
        colorscale=[[0, 'rgba(255,255,255,0.3)'], [1, 'rgba(255,0,0,0.7)']],
        zmin=0, zmax=gdf_clean['fctimp_mean'].max(),
        marker=dict(opacity=0.6, line=dict(width=2, color='yellow')),
        colorbar=dict(
            title="Total<br>Impervious",
            tickformat='.3f'
        ),
        hovertemplate='%{customdata}<extra></extra>',
        customdata=hover_text,
        name='Buffer Polygons',
        showlegend=True
    ))
    
    # ============ ADD SWOT NODES ============
    if nodes_df is not None and len(nodes_df) > 0:
        print(f"Adding {len(nodes_df)} SWOT nodes to map...")
        
        node_hover = []
        for _, row in nodes_df.iterrows():
            text = f"<b>SWOT Node</b><br>"
            text += f"Node ID: {row['node_id']}<br>"
            text += f"Reach ID: {row['reach_id']}<br>"
            text += f"Lat: {row['lat']:.4f}, Lon: {row['lon']:.4f}"
            node_hover.append(text)
        
        fig.add_trace(go.Scattermapbox(
            lon=nodes_df['lon'],
            lat=nodes_df['lat'],
            mode='markers',
            marker=dict(size=8, color='lime', symbol='circle', opacity=0.8),
            text=node_hover,
            hovertemplate='%{text}<extra></extra>',
            name='SWOT Nodes',
            showlegend=True
        ))
    
    # USGS gages
    if len(snapped_gages_df) > 0:
        gage_hover = []
        for _, row in snapped_gages_df.iterrows():
            text = f"<b>USGS Gage</b><br>"
            text += f"Site #{row['site_number']}<br>"
            text += f"Site No: {row['site_no']}<br>"
            text += f"Station: {row['station_nm']}"
            gage_hover.append(text)
        
        fig.add_trace(go.Scattermapbox(
            lon=snapped_gages_df['snapped_lon'],
            lat=snapped_gages_df['snapped_lat'],
            mode='markers',
            marker=dict(size=12, color='red', symbol='circle', opacity=1.0),
            text=gage_hover,
            hovertemplate='%{text}<extra></extra>',
            name='USGS Gages',
            showlegend=True
        ))
    
    # Dropdown buttons
    zoom_buttons = [dict(
        label='All Sites (US View)',
        method='relayout',
        args=[{'mapbox.center': {'lat': 39, 'lon': -98}, 'mapbox.zoom': 3.5}]
    )]
    
    for info in zoom_info:
        zoom_buttons.append(dict(
            label=f'Site #{info["site_number"]} - {info["reach_id"]}',
            method='relayout',
            args=[{'mapbox.center': {'lat': info['center_lat'], 'lon': info['center_lon']},
                   'mapbox.zoom': info['zoom']}]
        ))
    
    # KEY PART: USGS Imagery layer!
    fig.update_layout(
        mapbox=dict(
            style="white-bg",
            layers=[{
                "below": 'traces',
                "sourcetype": "raster",
                "sourceattribution": "United States Geological Survey",
                "source": [
                    "https://basemap.nationalmap.gov/arcgis/rest/services/USGSImageryOnly/MapServer/tile/{z}/{y}/{x}"
                ]
            }],
            center={'lat': 39, 'lon': -98},
            zoom=3.5
        ),
        updatemenus=[dict(
            buttons=zoom_buttons,
            direction='down',
            x=0.01, y=0.80,
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='cyan',
            borderwidth=2
        )],
        title='Selected 12 Reaches - USGS Satellite Imagery with SWOT Nodes',
        height=800,
        margin=dict(l=0, r=0, t=50, b=0),
        showlegend=True,
        legend=dict(
            x=0.01, y=0.99,
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=2
        )
    )
    
    if save_path:
        fig.write_html(save_path)
        print(f"Saved: {save_path}")
    
    return fig, snapped_gages_df, nodes_df


# Create map with USGS satellite imagery + SWOT nodes
print("Creating Map with USGS Satellite Imagery + SWOT Nodes")

fig_imagery, snapped_gages, nodes_df = create_sword_buffer_gage_map_imagery(
    selected_12,
    sword_shp_path='/work/pi_kandread_umass_edu/swot-urban/1_data_processing/reaches_v1.shp',
    usgs_csv_path="/work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/paired_reach_SWOT_gage.csv",
    swot_node_path='/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_swot_qc_filtered_vD_node.parquet',
    save_path='/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/map_usgs_imagery_with_nodes.html'
)

print("\nCreated map with USGS satellite imagery + SWOT nodes!")
fig_imagery

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.gridspec import GridSpec
import contextily as cx
from shapely.geometry import Point
from shapely.ops import nearest_points
import numpy as np
import warnings
warnings.filterwarnings('ignore')


def snap_point_to_line(point, line):
    """Snap a point to the nearest location on a line."""
    nearest = nearest_points(point, line)
    return nearest[1]


def create_paper_figures(selected_gdf, sword_shp_path, usgs_csv_path, 
                         swot_node_path=None, save_path=None, dpi=300):
    """
    Create publication-ready figures:
    - Row 1: US overview map with colorbar on right
    - Rows 2-5: 12 individual reach maps (4 rows x 3 columns)
    """
    
    print("Loading data...")
    
    # Load USGS gage locations
    usgs_loc = pd.read_csv(usgs_csv_path, dtype={'gage_id': object})
    usgs_loc = usgs_loc.rename(columns={'gage_id': 'site_no', 
                                        'gage_latitude': 'dec_lat_va',
                                        'gage_longitude': 'dec_long_va',
                                        'gage_name': 'station_nm'})
    
    # Load SWORD reaches
    sword_reaches = gpd.read_file(sword_shp_path).to_crs(epsg=4326)
    selected_reach_ids = selected_gdf['reach_id'].tolist()
    sword_selected = sword_reaches[sword_reaches['reach_id'].isin(selected_reach_ids)].copy()
    
    # Load SWOT node data if provided
    nodes_df = None
    if swot_node_path:
        print("Loading SWOT node data...")
        df_swot = pd.read_parquet(swot_node_path)
        nodes_df = df_swot[df_swot['reach_id'].isin(selected_reach_ids)].copy()
        if 'node_id' in nodes_df.columns:
            nodes_df = nodes_df.drop_duplicates(subset=['node_id'])[['node_id', 'reach_id', 'lat', 'lon']].copy()
            print(f"  Found {len(nodes_df)} unique nodes")
    
    # Load US state boundaries
    print("Loading state boundaries...")
    try:
        states = gpd.read_file(
            "https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_state_500k.zip"
        )
    except:
        world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
        states = world[world['name'] == 'United States of America']
    
    # Filter to continental US states
    continental_states = ['WA', 'OR', 'CA', 'NV', 'ID', 'MT', 'WY', 'UT', 'AZ', 'CO', 'NM',
                         'ND', 'SD', 'NE', 'KS', 'OK', 'TX', 'MN', 'IA', 'MO', 'AR', 'LA',
                         'WI', 'IL', 'MS', 'MI', 'IN', 'KY', 'TN', 'AL', 'OH', 'WV', 'VA',
                         'NC', 'SC', 'GA', 'FL', 'PA', 'NY', 'VT', 'NH', 'MA', 'RI', 'CT',
                         'NJ', 'DE', 'MD', 'ME', 'DC']
    
    if 'STUSPS' in states.columns:
        states = states[states['STUSPS'].isin(continental_states)].copy()
        states_3857 = states.to_crs(epsg=3857)
    else:
        states_3857 = None
    
    # Prepare data
    selected_gdf = selected_gdf.to_crs(epsg=4326).copy()
    
    if 'site_numbe' not in selected_gdf.columns:
        if 'Unnamed: 0' in selected_gdf.columns:
            selected_gdf['site_numbe'] = selected_gdf['Unnamed: 0'] + 1
        else:
            selected_gdf['site_numbe'] = range(1, len(selected_gdf) + 1)
    
    selected_gdf = selected_gdf.sort_values('site_numbe').reset_index(drop=True)
    
    # Snap USGS gages
    print("Snapping USGS gages...")
    snapped_gages = []
    for _, row in selected_gdf.iterrows():
        usgs_match = usgs_loc[usgs_loc['site_no'] == row['site_no']]
        if len(usgs_match) > 0:
            gage_point = Point(usgs_match.iloc[0]['dec_long_va'], usgs_match.iloc[0]['dec_lat_va'])
            sword_match = sword_selected[sword_selected['reach_id'] == row['reach_id']]
            if len(sword_match) > 0:
                snapped_point = snap_point_to_line(gage_point, sword_match.iloc[0].geometry)
                snapped_gages.append({
                    'site_no': row['site_no'],
                    'site_numbe': row['site_numbe'],
                    'station_nm': usgs_match.iloc[0]['station_nm'],
                    'snapped_lat': snapped_point.y,
                    'snapped_lon': snapped_point.x,
                    'reach_id': row['reach_id']
                })
    
    snapped_gages_df = pd.DataFrame(snapped_gages)
    
    # Convert to Web Mercator for plotting
    selected_gdf_3857 = selected_gdf.to_crs(epsg=3857)
    sword_selected_3857 = sword_selected.to_crs(epsg=3857)
    
    if nodes_df is not None:
        nodes_gdf = gpd.GeoDataFrame(
            nodes_df,
            geometry=gpd.points_from_xy(nodes_df['lon'], nodes_df['lat']),
            crs='EPSG:4326'
        ).to_crs(epsg=3857)
    
    if len(snapped_gages_df) > 0:
        gages_gdf = gpd.GeoDataFrame(
            snapped_gages_df,
            geometry=gpd.points_from_xy(snapped_gages_df['snapped_lon'], snapped_gages_df['snapped_lat']),
            crs='EPSG:4326'
        ).to_crs(epsg=3857)
    
    # Color normalization
    vmin = selected_gdf['fctimp_mea'].min()
    vmax = selected_gdf['fctimp_mea'].max()
    norm = Normalize(vmin=vmin, vmax=vmax)
    cmap = plt.cm.Reds
    
    # Create figure with GridSpec
    # 6 rows: overview, legend space, 4 rows of reaches
    # 4 columns: 3 for maps, 1 for colorbar
    
    fig = plt.figure(figsize=(12, 18))
    
    # GridSpec: 6 rows, 4 columns
    # Row 0: Overview map (cols 0-2) + colorbar (col 3)
    # Row 1: Legend (spans all)
    # Rows 2-5: Reach maps (3 per row)
    gs = GridSpec(6, 4, figure=fig, 
                  height_ratios=[1.2, 0.08, 1, 1, 1, 1],
                  width_ratios=[1, 1, 1, 0.05],
                  hspace=0.15, wspace=0.08)
    
    # Row 0: US Overview Map
    print("\nCreating US overview map...")
    ax_us = fig.add_subplot(gs[0, 0:3])  # First 3 columns
    
    # Set extent
    us_bounds = selected_gdf_3857.total_bounds
    x_range = us_bounds[2] - us_bounds[0]
    y_range = us_bounds[3] - us_bounds[1]
    padding_x = x_range * 0.12
    padding_y = y_range * 0.12
    
    ax_us.set_xlim(us_bounds[0] - padding_x, us_bounds[2] + padding_x)
    ax_us.set_ylim(us_bounds[1] - padding_y, us_bounds[3] + padding_y)
    
    # Add basemap
    try:
        cx.add_basemap(ax_us, source=cx.providers.Esri.WorldImagery, zoom=5, attribution="")
    except:
        cx.add_basemap(ax_us, source=cx.providers.CartoDB.Positron, zoom=5, attribution="")
    
    # Add state boundaries
    if states_3857 is not None:
        states_3857.boundary.plot(ax=ax_us, color='white', linewidth=0.8, alpha=0.7, zorder=2)
        
        for _, state in states_3857.iterrows():
            centroid = state.geometry.centroid
            if (us_bounds[0] - padding_x < centroid.x < us_bounds[2] + padding_x and
                us_bounds[1] - padding_y < centroid.y < us_bounds[3] + padding_y):
                ax_us.annotate(
                    state['STUSPS'],
                    xy=(centroid.x, centroid.y),
                    fontsize=7, fontweight='normal',
                    color='white', alpha=0.8,
                    ha='center', va='center',
                    zorder=3
                )
    
    # Plot points
    centroids_3857 = selected_gdf_3857.geometry.centroid
    scatter = ax_us.scatter(
        centroids_3857.x, centroids_3857.y,
        c=selected_gdf_3857['fctimp_mea'],
        cmap=cmap, norm=norm,
        s=100, edgecolor='black', linewidth=1.5, zorder=5
    )
    
    # Add reach numbers
    for idx, row in selected_gdf_3857.iterrows():
        centroid = row.geometry.centroid
        ax_us.annotate(
            f"{int(row['site_numbe'])}",
            xy=(centroid.x, centroid.y),
            xytext=(6, 6), textcoords='offset points',
            fontsize=9, fontweight='bold',
            color='black',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8, edgecolor='none'),
            zorder=6
        )
    
    ax_us.set_title('(a) Study Reaches', fontsize=14, fontweight='bold', pad=10)
    ax_us.axis('off')
    
    # Colorbar in column 3
    ax_cbar = fig.add_subplot(gs[0, 3])
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, cax=ax_cbar, orientation='vertical')
    cbar.set_label('Total Impervious Fraction', fontsize=10)
    cbar.ax.tick_params(labelsize=9)
    
    # Row 1: Legend
    ax_legend = fig.add_subplot(gs[1, :])
    ax_legend.axis('off')
    
    legend_elements = [
        mpatches.Patch(facecolor='none', edgecolor='yellow', linewidth=2, label='Buffer Polygon'),
        plt.Line2D([0], [0], color='cyan', linewidth=3, label='SWOT Reaches'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='lime', 
                   markeredgecolor='darkgreen', markersize=8, label='SWOT Nodes'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', 
                   markeredgecolor='darkred', markersize=10, label='USGS Gage')
    ]
    
    ax_legend.legend(handles=legend_elements, loc='center', ncol=4, 
                     fontsize=10, frameon=True, fancybox=True,
                     framealpha=0.9, edgecolor='gray')
    
    # Rows 2-5: Individual Reach Maps
    print("\nCreating individual reach maps...")
    
    selected_gdf_3857_sorted = selected_gdf_3857.sort_values('site_numbe').reset_index(drop=True)
    site_labels = ['b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm']
    
    for i, (idx, row) in enumerate(selected_gdf_3857_sorted.iterrows()):
        row_idx = (i // 3) + 2  # Rows 2-5
        col_idx = i % 3         # Columns 0-2
        
        ax = fig.add_subplot(gs[row_idx, col_idx])
        
        site_numbe = int(row['site_numbe'])
        reach_id = row['reach_id']
        
        print(f"  Reach #{site_numbe} (Reach ID: {reach_id})")
        
        # Set extent
        poly_bounds = row.geometry.bounds
        x_range = poly_bounds[2] - poly_bounds[0]
        y_range = poly_bounds[3] - poly_bounds[1]
        
        max_range = max(x_range, y_range)
        center_x = (poly_bounds[0] + poly_bounds[2]) / 2
        center_y = (poly_bounds[1] + poly_bounds[3]) / 2
        
        padding = max_range * 0.15
        half_size = (max_range / 2) + padding
        
        ax.set_xlim(center_x - half_size, center_x + half_size)
        ax.set_ylim(center_y - half_size, center_y + half_size)
        
        # Add basemap
        try:
            cx.add_basemap(ax, source=cx.providers.Esri.WorldImagery, zoom='auto', attribution="")
        except:
            pass
        
        # Plot buffer polygon
        selected_gdf_3857[selected_gdf_3857['site_numbe'] == site_numbe].plot(
            ax=ax, facecolor='none', edgecolor='yellow', linewidth=2, zorder=3
        )
        
        # Plot SWOT reaches
        sword_match = sword_selected_3857[sword_selected_3857['reach_id'] == reach_id]
        if len(sword_match) > 0:
            sword_match.plot(ax=ax, color='cyan', linewidth=3, zorder=4)
        
        # Plot SWOT nodes
        if nodes_df is not None:
            nodes_site = nodes_gdf[nodes_gdf['reach_id'] == reach_id]
            if len(nodes_site) > 0:
                nodes_site.plot(ax=ax, color='lime', markersize=20, zorder=5, 
                               marker='o', edgecolor='darkgreen', linewidth=0.5)
        
        # Plot USGS gage
        if len(snapped_gages_df) > 0:
            gage_site = gages_gdf[gages_gdf['site_numbe'] == site_numbe]
            if len(gage_site) > 0:
                gage_site.plot(ax=ax, color='red', markersize=80, zorder=6, 
                              marker='o', edgecolor='darkred', linewidth=1)
        
        ax.set_title(f'({site_labels[i]}) Reach {site_numbe}', 
                    fontsize=11, fontweight='bold', pad=5)
        ax.axis('off')
    
    plt.tight_layout()
    
    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight', facecolor='white')
        print(f"\nSaved: {save_path}")
        
        pdf_path = save_path.replace('.png', '.pdf')
        fig.savefig(pdf_path, dpi=dpi, bbox_inches='tight', facecolor='white')
        print(f"Saved: {pdf_path}")
    
    return fig


# Main execution
if __name__ == "__main__":
    print("Creating Publication-Ready Figures")
    
    selected_12_path = '/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12_vD.shp'
    
    try:
        selected_12 = gpd.read_parquet(selected_12_path)
    except:
        try:
            selected_12 = gpd.read_file(selected_12_path.replace('.parquet', '.geojson'))
        except:
            selected_12 = gpd.read_file(selected_12_path.replace('.parquet', '.shp'))
    
    print(f"Columns: {selected_12.columns.tolist()}")
    
    fig = create_paper_figures(
        selected_12,
        sword_shp_path='/work/pi_kandread_umass_edu/swot-urban/1_data_processing/reaches_v1.shp',
        usgs_csv_path="/work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/paired_reach_SWOT_gage.csv",
        swot_node_path='/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_swot_qc_filtered_vD_node.parquet',
        save_path='/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/paper_figure_study_reaches.png',
        dpi=300
    )
    
    print("\nDone!")
    plt.show()